[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r1/01_deg_analysis.ipynb)

# R1-Q1 Week 2 — Differential Expression

You're in Week 2 of R1-Q1. The orientation notebook introduced AtGenExpress — what it is, who built it, and what's in it. This week's job is to identify, for each stress in the dataset, which genes respond to that stress. The output is one gene list per (stress, tissue) pair — the building blocks Week 3 will intersect to find the common stress core.

By the end of this notebook you will have:

- A metadata table mapping each sample (GSM) to its stress, tissue, time point, treatment status, and biological replicate.
- Walked one differential expression comparison end-to-end (cold-shoot) so every analytical choice is explicit.
- A long-format DE results table covering all eight stresses across both tissues — sixteen comparisons in total.
- Documented decisions on time-point handling, statistical thresholds, and multiple-testing correction — the material EQ#2 asks you to write up.

In [ ]:
# First we install the base package, using the package manager `pip`
# pip will pull the irilab2026 package from GitHub and install it.
# We also install the dependencies `patsy` and `inmoose` which are used in the notebook.
# This cell will take about 45 seconds to run, as it needs to download and install the packages.

!pip install git+https://github.com/geraldmc/irilab2026.git -q

!pip install -q patsy inmoose

## 2) Setup and data

We start by calling `setup()` from the iRI Lab library. It standardizes the
environment between Colab and a local machine — checks for GPU, mounts Drive
if you want it, sets up the cache directory used by the data loaders. You'll
see this same call in every notebook in the program; this is the last
notebook that walks through what it does.

In [ ]:
import irilab2026 as iri

iri.setup()

Now load AtGenExpress. `load_atgenexpress()` is the function we built last
week. The first time it runs it downloads the nine GEO accessions (GSE5620
through GSE5628) and parses them into pandas DataFrames; subsequent runs
read from the cache and return immediately.

In [ ]:
data = iri.load_atgenexpress()
print(list(data.keys()))

Nine keys: one matched control (`control`) plus eight abiotic stresses
(`cold`, `osmotic`, `salt`, `drought`, `genotoxic`, `uv_b`, `wounding`,
`heat`). The ninth AtGenExpress stress — oxidative — is hosted at
TAIR/NASCArrays rather than at GEO, so it is not in this loader's output.
We work with the eight that are.

Look at one of these to see the shape of the data:

In [ ]:
cold = data['cold']
print(f"Shape: {cold.shape}  (probes × samples)")
cold.iloc[:5, :3]

The rows are Affymetrix ATH1 probe IDs — about 22,810 of them, covering
most of the *Arabidopsis* genome. The columns are GEO sample IDs (those
`GSM131xxx` strings). The cell values are normalized expression
intensities.

Notice what is *not* in this matrix: the column names tell us which sample
each measurement came from, but they do not tell us *what kind of sample*
— which tissue, which time point, which biological replicate. Without that
information we cannot run differential expression. The next section
recovers it.

## 3) Parsing the metadata

Sample-level information is encoded in the *titles* of the GEO samples —
human-readable strings attached to each GSM when the dataset was deposited
in 2007. The titles look like:

    AtGen_6-0011_Control-Shoots-0h_Rep1
    AtGen_6-0712_Cold-Roots-1h_Rep2
    AtGen_6-9999_Heat-Shoots-12h_Rep1

The format is consistent: an internal experiment code, then condition,
tissue, time point, and replicate number, all separated by hyphens or
underscores. Our job is to take these titles and produce a tidy table
indexed by GSM with one column per field.

We use GEOparse — the same library `load_atgenexpress()` uses internally
— to read the cached SOFT files and pull out each sample's title. No
network round-trip needed: the SOFT files are already on disk from the
loader's first run.

In [ ]:
import GEOparse
from pathlib import Path

# The loader puts SOFT files in the iRI Lab cache. Find them all.
soft_files = sorted(iri.cache_dir().rglob('*.soft.gz'))
for p in soft_files:
    print(p)

Nine SOFT files, one per GEO series. Open one of them and look at the
sample titles to confirm the format we expect:

In [ ]:
gse = GEOparse.get_GEO(filepath=str(soft_files[1]), silent=True)  # GSE5621 = cold

for gsm_id, gsm in list(gse.gsms.items())[:6]:
    title = ' '.join(gsm.metadata.get('title', []))
    print(f"{gsm_id}  {title}")

Six cold samples. The structure is exactly as expected: stress name
(`Cold`), tissue (`Shoots` or `Roots`), time point (in hours), replicate
(`Rep1` or `Rep2`).

### 3.1) Designing the parser

The strategy is the one the Pass 1 metadata script used: parse what is
parseable, flag what is not. Guessing on ambiguous fields would silently
mislabel samples, which is worse than leaving fields blank and dealing
with the gaps explicitly.

Two of the four fields need regex. Tissue is a simple keyword check
(`shoot` or `root`); stress comes directly from the GSE the sample
belongs to (no parsing needed); time and replicate need pattern matching
because their values vary.

The time regex is worth one paragraph because there is a real subtlety in
it. AtGenExpress titles use underscores after the time unit
(`0.25h_Rep1`). Python's regex word boundary `\\b` does not trigger
before an underscore — Python treats underscore as a word character — so
a naive `\\b` after the unit fails to match. The regex below uses an
explicit lookahead that accepts end-of-string, whitespace, underscore,
and common delimiters.

In [ ]:
import re

# Time point: matches '0h', '0.25h', '0.5 h', '30 min', etc.
TIME_RE = re.compile(
    r'(\d+(?:\.\d+)?)\s*'
    r'(hours?|hrs?|h|minutes?|mins?|m)'
    r'(?=$|[\s_\-,;|/])',
    re.IGNORECASE,
)

# Replicate: matches 'Rep1', 'replicate 2', '_R1', etc.
REP_RE = re.compile(
    r'(?:replicate|rep|biological\s*replicate)\D*?(\d+)'
    r'|_R(\d+)\b',
    re.IGNORECASE,
)

def extract_time(title):
    m = TIME_RE.search(title)
    if not m:
        return None
    value = float(m.group(1))
    unit = m.group(2).lower()
    return value / 60.0 if unit.startswith('m') else value

def extract_rep(title):
    m = REP_RE.search(title)
    if not m:
        return None
    for g in m.groups():
        if g:
            return int(g)
    return None

def extract_tissue(title):
    t = title.lower()
    if 'root' in t:
        return 'root'
    if any(w in t for w in ('shoot', 'leaf', 'leaves', 'rosette', 'aerial')):
        return 'shoot'
    return None

Quick sanity check on a few representative titles before we run across
the whole dataset:

In [ ]:
test_titles = [
    'AtGen_6-0011_Control-Shoots-0h_Rep1',
    'AtGen_6-0712_Cold-Roots-1h_Rep2',
    'AtGen_6-9999_Heat-Shoots-12h_Rep1',
    'AtGen_6-9999_Drought-Roots-0.25h_Rep2',
]

for t in test_titles:
    print(t)
    print(f"  tissue={extract_tissue(t)}  time={extract_time(t)}h  rep={extract_rep(t)}")

All four fields parse cleanly on a representative spread, including the
fractional time point (0.25h) and the underscore-after-unit case that
required the lookahead.

### 3.2) Apply across all nine GSEs

Stress comes from the GSE the sample belongs to. The mapping is fixed by
the dataset's design and lives in one dict:

In [ ]:
import pandas as pd

GSE_TO_STRESS = {
    'GSE5620': 'control',
    'GSE5621': 'cold',
    'GSE5622': 'osmotic',
    'GSE5623': 'salt',
    'GSE5624': 'drought',
    'GSE5625': 'genotoxic',
    'GSE5626': 'uv_b',
    'GSE5627': 'wounding',
    'GSE5628': 'heat',
}

rows = []
for gse_id, stress in GSE_TO_STRESS.items():
    soft_path = next(p for p in soft_files if gse_id in p.name)
    gse = GEOparse.get_GEO(filepath=str(soft_path), silent=True)
    for gsm_id, gsm in gse.gsms.items():
        title = ' '.join(gsm.metadata.get('title', []))
        rows.append({
            'GSM': gsm_id,
            'GSE': gse_id,
            'stress': stress,
            'tissue': extract_tissue(title),
            'time_h': extract_time(title),
            'replicate': extract_rep(title),
            'title': title,
        })

metadata = pd.DataFrame(rows).set_index('GSM')
print(f"Metadata rows: {len(metadata)}")
metadata.head()

248 rows, one per sample. Now validate.

### 3.3) Validation 1 — chip counts

The Pass 1 reality-check pinned the per-stress chip counts against
Hahn 2013 §4.1. If our parser is right, a groupby on `stress` should
reproduce them exactly:

In [ ]:
counts = metadata.groupby('stress').size()
expected = {
    'control': 36,
    'cold': 24, 'osmotic': 24, 'salt': 24, 'genotoxic': 24,
    'drought': 28, 'uv_b': 28, 'wounding': 28,
    'heat': 32,
}

for stress, n_obs in counts.items():
    n_exp = expected.get(stress, '?')
    match = 'OK' if n_obs == n_exp else f'MISMATCH (expected {n_exp})'
    print(f"  {stress:10s}  observed={n_obs:3d}   {match}")

All counts match. That's the strongest signal we have that the parser is
not silently mislabeling — Hahn 2013's chip counts come from a different
source (the published methods) and converge on the same numbers our
parser produced from the GEO metadata.

### 3.4) Validation 2 — time grid

The Pass 1 reality-check also found a clean six-point common core —
{0.5, 1, 3, 6, 12, 24} hours — shared by all eight stresses, plus a
0h baseline that lives only in the control series, and a few
stress-specific extra time points. A pivot table makes the grid
visible:

In [ ]:
time_grid = metadata.pivot_table(
    index='stress',
    columns='time_h',
    values='GSE',
    aggfunc='count',
    fill_value=0,
)
time_grid

The grid matches the reality-check exactly:

- `control` covers all nine time points (0, 0.25, 0.5, 1, 3, 4, 6, 12, 24h).
- `cold`, `osmotic`, `salt`, `genotoxic` share the 6-point core.
- `drought`, `uv_b`, `wounding` add 0.25h.
- `heat` adds 4h.
- The 0h baseline is in `control` only.

Two analytical consequences worth flagging now, both load-bearing for the
DE design in Sections 4 and 5:

1. **Stress samples have no within-stress 0h baseline.** When we compare
   stress to control, the comparison is "stress at time *t*" vs
   "control at time *t*" — same time point, different GSE — not "stress
   at time *t*" vs "stress at 0h."

2. **The 6-point core is the unit of cross-stress comparison.** When
   Week 3 intersects DE lists across stresses, the cleanest comparison
   uses the time points that all eight stresses share.

### Validation 3 — anything still flagged?

In [ ]:
unparsed = metadata[
    metadata['tissue'].isna()
    | metadata['time_h'].isna()
    | metadata['replicate'].isna()
]
print(f"Rows with missing fields: {len(unparsed)}")
unparsed.head() if len(unparsed) else None

Zero. Every sample parsed cleanly across all four fields.

### 3.5) One more move before Section 4

We will want to compare each stress sample to its matched control sample
(same tissue, same time point, same replicate). That comparison reads
more naturally if we add a `treatment` column — `True` if the sample
came from a stress GSE, `False` if it came from the control GSE.

In [ ]:
metadata['treatment'] = metadata['stress'] != 'control'
metadata['treatment'].value_counts()

212 stress samples (8 stresses × 2 tissues × varying time points × 2
replicates), 36 controls (1 control series × 2 tissues × 9 time points
× 2 replicates). The metadata DataFrame is what Section 4 will use as
the design table for the limma fit.

## 4) What "differential expression" means here

Section 3 produced a metadata table. The job now is to find, for each stress and each tissue, which genes respond to the stress. A gene "responds" if its expression intensities in stress samples differ more than chance would predict from its expression intensities in matched controls.

That comparison sounds straightforward — a t-test per gene — but two features of this dataset make it harder than that. First, only two biological replicates per condition (n=2), which leaves a per-gene t-test with two degrees of freedom and a variance estimate too unstable to trust. Second, each stress is sampled at six time points across the response, which means the question isn't "is this gene different at *one* moment" but "does this gene respond *across* the time course." Sections 4.2 and 4.3 address each of these in turn.

### 4.1) Why limma, and a note on the Python implementation

The standard tool for differential expression on small-n microarray data is **limma** (Smyth 2004). Limma's central move is *empirical Bayes moderation* of per-gene variance estimates: rather than estimating each gene's variance from its own two replicates alone, limma shrinks the per-gene variance toward a global prior estimated across all ~22,000 genes on the array. The moderated variance for a given gene is, schematically:

$$s^2_{\text{moderated}} = \frac{d_0 \, s_0^2 + d \, s_g^2}{d_0 + d}$$

where $s_g^2$ is the per-gene variance from this gene's replicates, $s_0^2$ is the global prior variance pooled across all genes, and $d_0, d$ are degrees of freedom for the prior and the gene respectively. The effect is that genes with extreme per-gene variances (high or low) get pulled toward the global average, which stabilizes the gene rankings and makes the lists we hand to Week 3 reproducible under reasonable threshold changes.

**A note on the Python implementation:** Limma is canonical in R/Bioconductor and most published microarray analyses use the R version. We use **InMoose** (Colange et al. 2025), a peer-reviewed Python port of limma that produces numerically identical output on microarray data — verified against the R original across 12 microarray datasets, with Pearson correlations of 1.0 for p-values and log-fold-changes. InMoose lets us stay in pure Python without sacrificing the method.

### 4.2) The time-course design

AtGenExpress samples each stress at six common time points after exposure (0.5, 1, 3, 6, 12, 24 hours). We could run a separate stress-vs-control t-test at each time point and take the union — "DE at any time" — but this approach has two problems: six tests per gene inflates the false-positive rate at any fixed significance threshold, and the union step doesn't account for multiple testing across time points.

The better approach uses limma's **moderated F-test**. We build a single design matrix encoding treatment status, time point, and their interaction, then ask one question per gene: *is there any effect of stress across this time course?* One test per gene, regardless of how many time points; captures transient early-response genes (CBF-family transcription factors peaking at 1-3 hours) and sustained late-response genes (COR genes, dehydrins) in the same test. This is what `eBayes()` plus `topTable()` will produce in Section 5: per-gene F-statistic, raw p-value, adjusted p-value, and the largest log-fold-change observed across the time course.

### 4.3) What you'll have at the end of Section 5

Section 5 walks one (stress, tissue) pair — cold-shoot — end-to-end with every analytical choice visible. The output is a DataFrame indexed by gene, with columns for the moderated F-statistic, raw p-value, adjusted p-value (Benjamini-Hochberg), and the maximum log-fold-change across time points.

Section 6 surfaces the threshold choices (adjusted-p cutoff, log-fold-change minimum) that turn the F-test output into a DE gene list — those are decisions you'll make and document for EQ#2, not defaults baked into a function call. Section 7 scales the analysis to all 16 (stress, tissue) pairs and produces a long-format table covering all of them; Section 8 sanity-checks the result against what we'd expect biologically.

## 5) An end-to-end comparison

Section 4 introduced limma and the moderated F-test conceptually. This section walks one (stress, tissue) comparison — cold-shoot — end-to-end on the actual data, with every analytical step visible. Once the pattern is clear here, Section 7 will scale it to all sixteen (stress, tissue) pairs.

We pick cold-shoot because cold is the most-studied stress in this dataset and shoot tissue (above ground) is where most of the well-known cold-responsive genes operate. That means we have a strong literature-based expectation for what the top hits should look like, which makes Section 5.10's sanity check meaningful.

### 5.1) Subset to cold-shoot samples and matched controls

A single differential expression comparison needs two groups: stress samples and matched controls. For cold-shoot specifically:

- **Stress side:** cold-stressed shoot samples at the six common time points (0.5, 1, 3, 6, 12, 24 hours). With two biological replicates each, that's 12 samples.
- **Control side:** control shoot samples at the same six time points. Again 12 samples.

The "matched" part is important. We don't pool all controls together; we pair each stress sample with its time-matched control. A cold-shoot-3h sample is compared against control-shoot-3h, not against a generic "control" average. This is how the time-course design we settled on in Section 4.3 actually gets executed.

In [ ]:
# Common time points across all eight stresses (from the reality check)
COMMON_TIMEPOINTS = [0.5, 1.0, 3.0, 6.0, 12.0, 24.0]

# Cold-shoot stress samples
cold_shoot_meta = metadata[
    (metadata['stress'] == 'cold')
    & (metadata['tissue'] == 'shoot')
    & (metadata['time_h'].isin(COMMON_TIMEPOINTS))
]

# Matched controls: control samples at the same time points, same tissue
control_shoot_meta = metadata[
    (metadata['stress'] == 'control')
    & (metadata['tissue'] == 'shoot')
    & (metadata['time_h'].isin(COMMON_TIMEPOINTS))
]

# Stack them — this is the sample set for the comparison
comparison_meta = pd.concat([cold_shoot_meta, control_shoot_meta])
print(f"Cold-shoot stress samples:    {len(cold_shoot_meta)}")
print(f"Cold-shoot matched controls:  {len(control_shoot_meta)}")
print(f"Total samples in comparison:  {len(comparison_meta)}")
comparison_meta[['stress', 'tissue', 'time_h', 'treatment', 'replicate']].sort_values(['treatment', 'time_h', 'replicate'])

24 samples in the comparison — 12 stress, 12 control, paired across the six common time points with two biological replicates each. The sorted table at the bottom should show two control samples at each of the six time points, then two cold samples at each of the same six time points. If you see anything else — missing time points, extra samples, a single replicate at some point — stop and check Section 3's metadata before going further.

Now pull the expression measurements for these samples. The cold-shoot stress measurements live in `data['cold']`, and the matched controls live in `data['control']`. We extract just the columns (samples) we need from each and stack them into one matrix.

In [ ]:
# Pull the expression columns for the samples we just selected.
# data['cold'] and data['control'] are gene × sample DataFrames, so we
# select columns (GSMs) by intersecting with our comparison metadata's index.
cold_cols = [gsm for gsm in cold_shoot_meta.index if gsm in data['cold'].columns]
ctrl_cols = [gsm for gsm in control_shoot_meta.index if gsm in data['control'].columns]

expr_matrix = pd.concat(
    [data['cold'][cold_cols], data['control'][ctrl_cols]],
    axis=1,
)

print(f"Expression matrix shape: {expr_matrix.shape}  (genes × samples)")
print(f"Columns are in this order:")
for gsm in expr_matrix.columns:
    row = comparison_meta.loc[gsm]
    print(f"  {gsm}  {row['stress']:8s}  t={row['time_h']:4.1f}h  rep={int(row['replicate'])}")

The matrix has ~22,810 rows (Affymetrix ATH1 probes) and 24 columns (samples). The column order matters for the next step — InMoose's `lmFit` will pair each column of the expression matrix to the corresponding row of the design matrix we're about to build, position by position. The print-out above lets you verify that order before we proceed.

### 5.2) Build the design matrix

A design matrix tells limma *what each sample represents* in the experiment. For our time-course comparison we need three pieces of information per sample:

- **Treatment status** — is this a stress sample or a control sample?
- **Time point** — at which of the six time points was this sample taken?
- **The interaction between them** — does the stress effect *change* across time points?

The interaction term is what makes this a time-course analysis rather than six independent comparisons. It encodes "the difference between stress and control may be different at 0.5h than at 24h," which is exactly the biology we expect (CBF transcription factors peak early; downstream COR genes peak later). When we run the moderated F-test in Section 5.9, the test asks: *is there any combination of treatment and time-point-specific treatment effect that differs from zero?* — and the design matrix is what defines what "any combination" means.

We build the design matrix with `patsy`, which is the standard Python tool for converting a metadata DataFrame into a design matrix using R-style formulas.

In [ ]:
import patsy

# Build the design matrix from the comparison metadata.
# Column order must match expr_matrix, so we reindex the metadata to match.
design_meta = comparison_meta.loc[expr_matrix.columns].copy()
design_meta['treatment'] = design_meta['treatment'].astype(int)  # bool -> 0/1
design_meta['time_factor'] = design_meta['time_h'].astype('category')

# Formula: treatment * time_factor expands to:
#   intercept + treatment + time_factor + treatment:time_factor
# That's the main treatment effect, the main time effect, and the interaction.
design = patsy.dmatrix(
    'treatment * time_factor',
    data=design_meta,
    return_type='dataframe',
)

print(f"Design matrix shape: {design.shape}  (samples × design terms)")
design.head()

The design matrix has 24 rows (one per sample) and 12 columns (one intercept + one treatment indicator + five time-factor indicators + five treatment-by-time interaction terms). The number of columns equals the degrees of freedom limma will use to model the experiment; the remaining 12 degrees of freedom (24 − 12) are the residual the moderated variance gets estimated from.

A note on what's *not* in the design matrix: biological replicates. Limma treats replicates as the source of residual variation rather than as a design factor. Each gene's two replicates per condition contribute to the residual error term, which is what makes the per-gene variance estimable in the first place. This is the n=2 problem from Section 4.2 made operational — with two replicates, the residual has just enough information to estimate variance per gene, and `eBayes` will stabilize that estimate by borrowing strength across all 22,810 genes.

### 5.3) Look at the expression matrix before fitting

Before we hand the expression matrix to limma, look at the distribution of values. This is a habit worth keeping for any new dataset: before running an analysis, confirm the data is on the scale the analysis expects.

In [ ]:
import numpy as np

print(f"Expression matrix value statistics:")
print(f"  min:    {expr_matrix.values.min():.2f}")
print(f"  median: {np.median(expr_matrix.values):.2f}")
print(f"  mean:   {expr_matrix.values.mean():.2f}")
print(f"  max:    {expr_matrix.values.max():.2f}")
print(f"  std:    {expr_matrix.values.std():.2f}")
print()
print(f"Are negative values present? {(expr_matrix.values < 0).any()}")
print(f"How many values > 100?       {(expr_matrix.values > 100).sum()}")
print(f"How many values > 1000?      {(expr_matrix.values > 1000).sum()}")

### 5.4) Log-transform the expression matrix

The output above tells us something important. Values range from 0 to roughly 11,000, with the median around 37 and a long right tail of highly expressed genes. No negative values. That's the signature of **linear-scale MAS5-normalized intensities**, which is how AtGenExpress was deposited in GEO and what the loader returned.

Limma — like nearly all standard differential expression methods — assumes its input is on a **log scale**. Log-transformation does three things the analysis depends on:

1. **It makes the variance more uniform across the expression range.** On linear scale, highly expressed genes have huge absolute variance just from being highly expressed, even when they aren't responding to anything. The empirical Bayes variance moderation we discussed in 4.2 does not work properly when most of the variance is just baseline-expression noise.

2. **It converts multiplicative effects (fold-changes) into additive ones.** Biology operates in fold-changes: "this gene goes up 4-fold under stress." Linear-scale differences confound fold-changes with baseline expression. Log-scale differences *are* fold-changes.

3. **It makes the data approximately normal.** Most expression distributions are heavily right-skewed on linear scale; log-transformation pulls the tail in and makes the per-gene values much closer to the normal distribution that the linear model assumes.

We use `log2(x + 1)` rather than `log2(x)` to handle the zero values cleanly. The `+1` shifts everything by a single unit, which is negligible for highly expressed genes (where the addition is invisible at log scale) but prevents `log(0)` errors for genes that read as zero on the array.

In [ ]:
# Log2-transform with pseudocount to handle zeros
expr_matrix_log = np.log2(expr_matrix + 1)

print(f"Log-transformed expression matrix value statistics:")
print(f"  min:    {expr_matrix_log.values.min():.2f}")
print(f"  median: {np.median(expr_matrix_log.values):.2f}")
print(f"  mean:   {expr_matrix_log.values.mean():.2f}")
print(f"  max:    {expr_matrix_log.values.max():.2f}")
print(f"  std:    {expr_matrix_log.values.std():.2f}")
print()
print(f"Now values should be roughly [0, 14] with median around 5.")
print(f"Same matrix shape: {expr_matrix_log.shape}")

### 5.5) Fit the linear model with `lmFit`

Now we fit the linear model. `lmFit` from InMoose fits a linear model per gene using the design matrix. The output is a fit object containing per-gene coefficients (one per design term) and per-gene residual variances — the raw material for the moderation step.

Note that we pass `expr_matrix_log`, not `expr_matrix`. Every cell from here on uses the log-transformed matrix.
5.5) Fit the linear model with lmFit
Now we fit the linear model. lmFit from InMoose fits a linear model per gene using the design matrix. The output is a fit object containing per-gene coefficients (one per design term) and per-gene residual variances — the raw material for the moderation step.

Note that we pass expr_matrix_log, not expr_matrix. Every cell from here on uses the log-transformed matrix.

In [ ]:
from inmoose import limma

fit = limma.lmFit(expr_matrix_log, design)
print(f"Fit object type: {type(fit).__name__}")
print(f"Coefficient matrix shape: {fit.coefficients.shape}  (genes × design terms)")
print(f"Per-gene residual variance shape: {fit.sigma.shape}  (genes,)")

### 5.6) Moderate the variance estimates with `eBayes`

`eBayes` computes a global prior variance from across all genes, then shrinks each gene's per-gene variance toward that prior. The output is a moderated residual variance per gene, plus the moderated F-statistic and p-value for the test we will use to rank genes in the next step.

Let's run it and compare a handful of genes' raw vs. moderated variances to see the shrinkage at work.

In [ ]:
fit_eb = limma.eBayes(fit)

# Compare raw vs. moderated variance for the first few genes
sigma_raw = fit.sigma ** 2          # raw per-gene variance (sigma is std dev)
sigma_mod = fit_eb.s2_post          # moderated posterior variance
prior_var = fit_eb.s2_prior         # the global prior

print(f"Global prior variance (s2_prior): {prior_var:.4f}\n")
print(f"{'Gene':<15s} {'Raw variance':>15s} {'Moderated':>15s} {'Pulled toward prior':>22s}")
for gene in expr_matrix_log.index[:8]:
    raw = sigma_raw.loc[gene]
    mod = sigma_mod.loc[gene]
    direction = "down" if mod < raw else "up"
    print(f"{gene:<15s} {raw:>15.4f} {mod:>15.4f} {direction:>22s}")

The "Pulled toward prior" column should show a mix of "up" and "down" directions: genes that happened to have low raw variance get pulled up toward the global prior, genes with high raw variance get pulled down. This is the equation from Section 4.2 in action — each gene's variance is now a weighted blend of its own information and the global prior, not its own information alone.

This is the move that makes downstream gene rankings stable. Without moderation, a gene whose two replicates happened to land very close together would get an artificially small variance, an artificially large t-statistic, and an artificially small p-value — appearing significant for the wrong reason. Moderation fixes this.

### 5.6) Extract the results with `topTable`

`topTable` formats the moderated F-test results into a per-gene table. For each gene we get the moderated F-statistic, raw p-value, adjusted p-value (Benjamini-Hochberg), and the largest absolute log-fold-change observed across the time-course coefficients.

In [ ]:
# InMoose renames coefficient columns to 'column0', 'column1', ... inside lmFit.
# We find which positions correspond to treatment-related terms in the
# original design matrix, then pass the renamed names ('column6', etc.)
# to topTable.
treatment_term_names = [c for c in design.columns if 'treatment' in c.lower()]
treatment_term_positions = [design.columns.get_loc(c) for c in treatment_term_names]
renamed_treatment_coefs = [f'column{i}' for i in treatment_term_positions]

print(f"Testing these coefficients jointly:")
for pos, orig_name, new_name in zip(treatment_term_positions, treatment_term_names, renamed_treatment_coefs):
    print(f"  position {pos:2d}: {orig_name}  ->  {new_name}")
print()

results = limma.topTable(
    fit_eb,
    coef=renamed_treatment_coefs,
    number=len(expr_matrix_log),
    sort_by='F',
)

# Rename to match what Section 4.4 promised
results = results.rename(columns={
    'F': 'F_stat',
    'pvalue': 'p_raw',
    'adj_pvalue': 'p_adj',
})

# Max |logFC| across the treatment coefficients, looked up by the renamed names
coef_values = fit_eb.coefficients[renamed_treatment_coefs]
results['max_abs_logfc'] = coef_values.abs().max(axis=1)

print(f"Results table shape: {results.shape}")
print(f"Results table columns: {list(results.columns)}\n")
results[['F_stat', 'p_raw', 'p_adj', 'max_abs_logfc']].head(10)

The top of the table shows the genes most strongly responsive to cold stress in shoot tissue, ranked by moderated F-statistic. Three columns to read:

- **F_stat** — the moderated F-statistic for "is any treatment-related coefficient non-zero." Larger means a stronger overall response across the time course.
- **p_adj** — the Benjamini-Hochberg adjusted p-value. This is the column to threshold on when Section 6 makes the cutoff decision.
- **max_abs_logfc** — the largest absolute log-fold-change across the time-course coefficients. A gene with high F-statistic but small log-fold-change is statistically reliable but biologically small; a gene with both is what we're really looking for.

### 5.7) Sanity check against known cold-responsive genes

The cold response in *Arabidopsis* has been studied for thirty years and has a well-known core. Three transcription factors — CBF1, CBF2, CBF3 — activate within an hour of cold exposure and turn on the COR ("cold-regulated") gene battery downstream. COR15a, COR47, and RD29A (also called COR78) are the most-cited downstream markers.

If our analysis is working, these six genes should all rank highly. If they don't, something is wrong upstream and we shouldn't trust the rest of the table.

In [ ]:
# Known cold-responsive marker genes, with their Affymetrix ATH1 probe IDs
# looked up against the GPL198 annotation. AGI loci shown for reference;
# the lookup against the results table uses probe IDs since that's how
# the expression matrix is keyed.
cold_markers = {
    'CBF1':   ('AT4G25490', '254074_at'),
    'CBF2':   ('AT4G25470', '254075_at'),
    'CBF3':   ('AT4G25480', '254066_at'),
    'COR15a': ('AT2G42540', '263497_at'),
    'COR47':  ('AT1G20440', '259570_at'),
    'RD29A':  ('AT5G52310', '248337_at'),
}

# Rank each marker in the results table (results is already sorted by F-statistic)
results_with_rank = results.copy()
results_with_rank['rank'] = range(1, len(results_with_rank) + 1)

print(f"{'Gene':<8s} {'AGI locus':<12s} {'Probe':<12s} {'Rank':>6s} {'F_stat':>10s} {'p_adj':>12s} {'max_logFC':>12s}")
print('-' * 76)

for gene_name, (agi, probe_id) in cold_markers.items():
    if probe_id in results_with_rank.index:
        row = results_with_rank.loc[probe_id]
        print(f"{gene_name:<8s} {agi:<12s} {probe_id:<12s} "
              f"{int(row['rank']):>6d} {row['F_stat']:>10.2f} "
              f"{row['p_adj']:>12.2e} {row['max_abs_logfc']:>12.2f}")
    else:
        print(f"{gene_name:<8s} {agi:<12s} {probe_id:<12s} "
              f"NOT FOUND in results")

What we expect to see if the analysis is working: all six cold markers ranking in the top hundred or so out of 22,810 genes, with `p_adj` values well below any reasonable threshold, and `max_logFC` values in the 2-to-7 range (which corresponds to 4-fold to 128-fold induction on the linear scale). The CBF transcription factors are the earliest responders and often the most dramatic — they frequently show the largest fold-changes in any cold experiment in *Arabidopsis*. The COR genes peak later but still show large effects.

If the markers come out where expected, the analysis is producing biologically meaningful output and we can move on to Section 6 (threshold decisions). If they don't — markers ranked low, small `max_logFC` values, or `p_adj > 0.05` — something is wrong upstream and we should look at the sample selection, the design matrix, or the data transformation before going further.

A small but useful detail: the `agi_to_probe` dictionary built when we fetched the GPL198 annotation is worth keeping around. Section 8 will use it to look up other marker genes when we sanity-check stresses we haven't walked end-to-end.

### 5.8) Reading the rankings: which sort matches the biology?

Look at the two bar charts above. The same six canonical cold-response genes (CBF1, CBF2, CBF3, COR15a, COR47, RD29A) appear in both. Their ranks are very different.

- **Left panel — ranked by F-statistic.** The markers sit between rank 3,000 and 14,000 out of 22,810. CBF1, CBF2, and CBF3 sit at almost identical ranks — that's because they're paralogous genes whose response shapes across the time course look statistically similar to the test. The F-statistic measures how reliably a gene differs between stress and control across the time course, but it doesn't reward effect *size*. A gene with small but extremely consistent differences can produce a larger F-statistic than a gene with one dramatic spike.

- **Right panel — ranked by max |logFC|.** All six markers land in the top 100. This is the ranking that matches biological intuition: the genes with the largest response sizes appear at the top.

The takeaway for differential expression analysis on this dataset:

> **F-statistic for testing, log-fold-change for ranking.** Use the F-statistic (and its associated p-value) to ask "is this gene significantly affected at all?" Use the max |logFC| to ask "of the genes that are significantly affected, which had the biggest response?"

Both questions matter, and the answers are not the same gene list. This is the substrate Section 6 will work with.

### 5.9) What "significant" looks like at this scale

Before we commit to thresholds in Section 6, look at the shape of two distributions: the raw p-values, and the max |logFC| values, across all 22,810 genes.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Left: raw p-value distribution
sns.histplot(
    results['p_raw'],
    bins=50,
    color='steelblue',
    edgecolor='black',
    ax=axes[0],
)
axes[0].set_xlabel('Raw p-value')
axes[0].set_ylabel('Number of genes')
axes[0].set_title('Distribution of raw p-values')

# Right: max |logFC| distribution
sns.histplot(
    results['max_abs_logfc'],
    bins=50,
    color='steelblue',
    edgecolor='black',
    ax=axes[1],
)
axes[1].set_xlabel('Max |logFC|')
axes[1].set_ylabel('Number of genes')
axes[1].set_title('Distribution of max |logFC| values')

plt.tight_layout()
plt.show()

# Companion summary
n_total = len(results)
n_sig = (results['p_adj'] < 0.05).sum()
n_lfc1 = (results['max_abs_logfc'] >= 1).sum()
n_lfc2 = (results['max_abs_logfc'] >= 2).sum()
n_both1 = ((results['p_adj'] < 0.05) & (results['max_abs_logfc'] >= 1)).sum()
n_both2 = ((results['p_adj'] < 0.05) & (results['max_abs_logfc'] >= 2)).sum()

print(f"Total genes:                                {n_total:>7,}")
print(f"Genes with p_adj < 0.05:                    {n_sig:>7,}  ({100*n_sig/n_total:.1f}%)")
print(f"Genes with max |logFC| >= 1 (2-fold):       {n_lfc1:>7,}  ({100*n_lfc1/n_total:.1f}%)")
print(f"Genes with max |logFC| >= 2 (4-fold):       {n_lfc2:>7,}  ({100*n_lfc2/n_total:.1f}%)")
print(f"p_adj < 0.05 AND max |logFC| >= 1:          {n_both1:>7,}")
print(f"p_adj < 0.05 AND max |logFC| >= 2:          {n_both2:>7,}")

The two distributions tell very different stories.

**The p-value distribution is degenerate.** A well-behaved differential expression result shows a sharp spike near zero (real DE genes) *and* a roughly flat plateau from p ≈ 0.1 to p ≈ 1.0 (null genes uniformly distributed by chance). Here there is no plateau. Essentially every gene has a raw p-value at or near zero. The summary below the plots makes this concrete: 99.9% of genes pass the `p_adj < 0.05` threshold.

This is not a bug. It's a real property of moderated tests applied to small experiments with strong consistent signals. AtGenExpress has only 12 residual degrees of freedom in this comparison, but limma's empirical Bayes step borrows information across all 22,810 genes to produce extraordinarily sensitive variance estimates. The result is a test that can detect tiny consistent differences as "statistically significant" — including differences too small to matter biologically.

**The log-fold-change distribution is well-behaved.** Most genes show very small response (mode around |logFC| ≈ 0.7), with a long right tail of strongly-responsive genes extending out past |logFC| ≈ 6. This is a distribution that *can* discriminate — pick a threshold and you get a meaningful gene list.

**The practical takeaway for this dataset:**

> Statistical significance is essentially universal here, which means the p-value filter is doing almost no work. **The effect-size threshold (`max_abs_logfc`) is what actually decides what counts as differentially expressed.**

This shapes everything in Section 6. The threshold decisions to document for EQ#2 are not about "what p_adj cutoff" — they are about "what minimum biological effect size is large enough to call this gene a cold-response gene." The p-value filter still belongs in the analysis (it's a formal gate against very weak signals), but the logFC threshold is doing the discriminating.

### 5.10) What we have, and what's next

For one (stress, tissue) pair we now have:

- A per-gene results table with moderated F-statistic, raw and adjusted p-values, and max |logFC| across the time course.
- Visual confirmation that the analysis is producing biologically reasonable output (six canonical cold markers all in the top 100 by |logFC|).
- Honest visibility into where statistical significance and biological importance diverge on this dataset.

Section 6 will use this groundwork to settle the threshold decisions you'll document for EQ#2: what minimum max |logFC| counts as a meaningful response, what role the `p_adj` filter still plays even when it's nearly universal, and how to apply Benjamini-Hochberg correction across the 16 (stress, tissue) pairs (since each pair is its own multiple-testing problem). Section 7 then wraps the pattern from Section 5 into a function and applies it across all 16 pairs to produce the long-format DE table that Week 3's intersection notebook will consume.